## Estructura del mapa.
Pensaremos el entorno como una cuadricula de celdas (una matriz de python NxM).

El rover podra moverse en 4 direcciones (arriba, abajo, izquierda, derecha) o en 8 direcciones (incluyendo diagonales).

### Los Tipos de Terreno y sus Costos (Batería)

Cada tipo de terreno tendrá un "peso" o costo energético asociado

* **Terreno Liso (o Pista):** Es una superficie plana y fácil (como asfalto o roca sólida).
* *Costo de batería:* Muy bajo (`1` unidad de batería).


* **Terreno Rugoso (o con Rocas):** Hay piedras sueltas que obligan al rover a maniobrar despacio y forzar los motores.
* *Costo de batería:* Moderado (`3` unidades).


* **Arena Fina:** El rover se hunde un poco y las ruedas patinan.
* *Costo de batería:* Alto (`5` unidades).


* **Barro:** La peor pesadilla del rover; se atora y requiere mucha potencia salir de ahí.
* *Costo de batería:* Muy alto (`8` unidades).


* **Obstáculo Impasable (Paredes o abismos):** Bloques por los que el rover **no puede pasar** bajo ningún concepto.
* *Costo de batería:* Infinito o bloqueado (el algoritmo de búsqueda sabrá que por ahí no se puede ir).


### Otros elementos que debe tener el entorno

Además de las casillas con sus costos de batería, el documento menciona que necesitamos simular otros elementos esenciales para que la misión tenga gracia:

* **Puntos de Interés (POI):** Casillas específicas del mapa donde hay "algo científico interesante" (una roca rara, un cráter, un fósil). El rover tiene que pasar por ahí para recolectar datos. Cada POI podría tener una puntuación o valor científico diferente (ej. visitar el POI A da 50 puntos, el POI B da 100 puntos).
* **Zonas de Comunicación:** Casillas especiales en el mapa. Como el rover tiene una **memoria limitada** (se le llena el disco duro guardando fotos y datos de los POI), necesita llegar de vez en cuando a estas zonas para "transmitir" la información a la base y liberar espacio en su memoria.


* **Estación de Recarga (Opcional, pero muy útil):** Una casilla donde el rover puede recargar su batería al 100% (o recuperar energía), porque si la batería llega a 0, la misión fracasa.





### 3. ¿Qué más elementos clave debe tener el entorno?

Además de las casillas con sus costos de batería, el documento menciona que necesitamos simular otros elementos esenciales para que la misión tenga gracia:

* **Puntos de Interés (POI):** Casillas específicas del mapa donde hay "algo científico interesante" (una roca rara, un cráter, un fósil). El rover tiene que pasar por ahí para recolectar datos. Cada POI podría tener una puntuación o valor científico diferente (ej. visitar el POI A da 50 puntos, el POI B da 100 puntos).
* **Zonas de Comunicación:** Casillas especiales en el mapa. Como el rover tiene una **memoria limitada** (se le llena el disco duro guardando fotos y datos de los POI), necesita llegar de vez en cuando a estas zonas para "transmitir" la información a la base y liberar espacio en su memoria.


* **Estación de Recarga (Opcional, pero muy útil):** Una casilla donde el rover puede recargar su batería al 100% (o recuperar energía), porque si la batería llega a 0, la misión fracasa.



---

### 4. ¿Cómo se vería esto en código mentalmente?

Para que luego sea fácil de generar con un script por **semillas** (como pide el profesor), puedes pensar en tu mapa como una matriz de números o un diccionario de celdas:

* `0` = Terreno liso (Costo: 1)
* `1` = Terreno rugoso (Costo: 3)
* `2` = Arena fina (Costo: 5)
* `3` = Barro (Costo: 8)
* `-1` = Obstáculo/Pared
* `P` = Punto de Interés (POI)
* `C` = Zona de Comunicación

### El Generador por Semillas (La clave secreta)

El profesor exige que el entorno sea parametrizable mediante un generador de instancias basado en una **semilla** (*seed*).

* **¿Qué significa esto en la vida real?** Piensa en los juegos de supervivencia como *Minecraft*. Cuando pones una semilla numérica (por ejemplo, `12345`), el juego genera siempre el mismo mundo exacto con las mismas montañas, ríos y árboles. Si pones la semilla `99999`, te genera otro mundo completamente distinto pero reproducible.
* En Python, usarás la biblioteca `random` o `numpy` fijando la semilla (`random.seed(seed)`) antes de colocar aleatoriamente los obstáculos, los tipos de terreno y los POI en la cuadrícula. Así, cuando hagas tus experimentos (Módulo 3), podrás correr los algoritmos una y otra vez sobre los *mismos* mapas exactos para comparar quién es mejor.



Vamos a dar el siguiente gran salto en nuestro proyecto. Ya tenemos nuestro tablero de juego (el entorno parametrizable por semillas), y ahora nos toca construir el cerebro analítico: **El Módulo de Búsqueda**.

Si revisamos nuestra base teórica, un problema de búsqueda formalmente requiere definir ciertos elementos clave: el espacio de estados, el estado inicial, las acciones posibles, la función de transición de estados (`RESULT`), y la función de costo de camino. Para nuestro rover, esto significa traducir cada celda del mapa en un estado que el algoritmo puede recorrer de manera inteligente para evitar gastar batería de más.

---

### ¿Qué algoritmo usaremos? ¡A* (A-Star)!

En la teoría de búsqueda vimos que existen algoritmos ciegos (como BFS o Costo Uniforme) que exploran a ciegas en todas direcciones. Sin embargo, el rey indiscutible cuando tenemos un objetivo claro y costos variables es **A***.

A* utiliza una función de evaluación $f(n) = g(n) + h(n)$, donde:

1. **$g(n)$**: Es lo que ya gastamos de batería desde el inicio hasta la celda actual.


2. **$h(n)$**: Es una estimación inteligente (heurística) de lo que nos falta por recorrer hasta la meta. Usaremos la **distancia Manhattan** ($\vert{}\Delta y\vert{} + \vert{}\Delta x\vert{}$), que es una heurística admisible y consistente porque nunca sobreestima el costo real en una cuadrícula con movimientos ortogonales.



### Explicación de la analogía con la vida real:

* **La Cola con Prioridad (`heapq`)**: Imagina que estás buscando el camino más barato en una aplicación de mapas. El algoritmo guarda todas las rutas alternativas que va descubriendo en una lista de pendientes, pero siempre saca a revisar primero la que **pinta mejor** (menor costo acumulado + menor distancia estimada).


* **La Heurística de Manhattan**: Es como mirar al horizonte y calcular en línea recta cuántas cuadras faltan para llegar a la meta, sabiendo que las calles reales (los terrenos liso, rugoso o con barro) te obligarán a dar pequeños desvíos.



¿Qué tal te parece? Pruébalo juntando ambos scripts y dime si logras ver la ruta calculada por la terminal. Cuando estés listo, avanzamos al siguiente módulo clave del proyecto.


# Documentación del Proyecto: Módulo de Representación Simbólica

## Elección de la Representación y Justificación Teórica

### Definición del Problema

El objetivo de este módulo es dotar al rover de la capacidad de razonar sobre las acciones necesarias para cumplir con los objetivos de la misión (como navegar entre ubicaciones, recolectar muestras, analizarlas y transmitirlas a la base). Para abordar este desafío desde el punto de vista de la Inteligencia Artificial, se evaluaron dos paradigmas principales estudiados en el curso: **Los Problemas de Satisfacción de Restricciones (CSP)** y la **Planificación Clásica**.


### Análisis comparativo y Justificación de la Decisión

#### ¿Por qué se descarta un enfoque basado en CSP?

* **Naturaleza de las variables:** Un Problema de Satisfacción de Restricciones (CSP) requiere conocer de antemano y de forma estricta un número fijo de variables y decisiones que conformarán el espacio de soluciones del problema.
* **Rigidez ante la longitud del plan:** En la navegación y operación del rover, **no sabemos de antemano cuántas acciones formará la respuesta final**. Dependiendo de la topografía del terreno, la distribución de las muestras y la estrategia de la misión, el plan óptimo puede constar de 5, 11 o cualquier cantidad variable de acciones.
* **Limitación estructural:** Forzar el problema del rover a un esquema CSP exigiría acotar arbitrariamente un límite máximo de pasos (por ejemplo, asumir un plan fijo de exactamente 15 acciones), lo que genera ineficiencia computacional, incapacidad de adaptación y problemas de escalabilidad si la solución requiere más o menos pasos de los previstos.

#### ¿Por qué se selecciona la Planificación Clásica?

* **Horizonte dinámico:** Tal como indican los materiales del curso, si lo que importa es determinar **qué acciones realizar y en qué orden** sin conocer su cantidad exacta de antemano, la herramienta conceptual y algorítmica adecuada es la planificación.
* **Flexibilidad en la secuencia:** Los formalismos de planificación (como PDDL) permiten definir un estado inicial y una meta abierta, dejando que el motor de búsqueda determine dinámicamente la longitud y el camino exacto del plan.
* **Manejo del Estado y Acciones:** Permite modelar de forma limpia los operadores del rover (mover, tomar, analizar, transmitir) mediante esquemas genéricos con precondiciones y efectos, resolviendo de manera natural el problema del marco (*frame problem*).

-

Se selecciona formalmente la **Planificación Clásica** como el paradigma rector para la representación simbólica del rover. Esta elección garantiza la flexibilidad necesaria para resolver misiones con secuencias de longitud variable y sienta las bases para utilizar planificadores automatizados basados en heurísticas independientes del dominio.


##  Modelado del Dominio en PDDL (Tipos, Predicados y Operadores)

### 1. Descripción del Commit

En este commit se define formalmente el dominio del problema mediante el archivo **`rover-dominio.pddl`**. Esto incluye la declaración de los tipos de objetos del entorno, los predicados que componen el estado del mundo bajo la suposición de mundo cerrado, y los esquemas de acción que describen cómo evoluciona el sistema.

---

### 2. Elementos del Dominio Modelados

#### A. Tipos de Objetos (`:types`)

Para estructurar el dominio de forma limpia y reutilizable, se definen los tipos base necesarios para la misión:

* `lugar`: Representa las posiciones o nodos espaciales en los que puede encontrarse el rover o las muestras.
* `muestra`: Representa los elementos u objetos científicos que el rover debe recolectar, analizar y reportar.

#### B. Predicados (`:predicates`)

Los predicados describen las propiedades y relaciones que pueden ser verdaderas o falsas en un momento dado.

* **Suposición de Mundo Cerrado:** Se asume que cualquier hecho atómico que no esté explícitamente contenido en el conjunto de estados se considera falso por defecto.
* Los predicados implementados son:
* `(en ?l - lugar)`: Indica que el rover se encuentra actualmente en el lugar `?l`.
* `(conectado ?l1 - lugar ?l2 - lugar)`: Indica que existe un camino transitable entre el lugar `?l1` y el lugar `?l2`.
* `(muestra-en ?m - muestra ?l - lugar)`: Indica que la muestra `?m` se encuentra ubicada en el lugar `?l`.
* `(tiene ?m - muestra)`: Indica que el rover tiene en su poder (ha recolectado) la muestra `?m`.
* `(analizada ?m - muestra)`: Indica que la muestra `?m` ya ha sido procesada y analizada.
* `(transmitida ?m - muestra)`: Indica que la información de la muestra `?m` ha sido enviada exitosamente a la base.
* `(zona-com ?l - lugar)`: Indica que el lugar `?l` cuenta con una antena o zona de comunicación válida para transmitir datos.



#### C. Esquemas de Acción y Resolución del Problema del Marco

Para modelar las acciones del rover (`mover`, `tomar`, `analizar`, `transmitir`), se definen sus parámetros, sus **precondiciones** (`:precondition`) que deben cumplirse obligatoriamente para poder ejecutar la acción, y sus **efectos** (`:effect`) que modifican el estado añadiendo nuevos hechos o eliminando los obsoletos mediante el operador `not`.

* Esto resuelve de forma elegante el **problema del marco** (*frame problem*), ya que el planificador no necesita evaluar qué propiedades del mundo *permanecen iguales*, sino que únicamente actualiza de manera explícita aquello que cambia con cada operador.



## Análisis de los resultados

1. Análisis del Parsing y Grounding
Carga exitosa: El dominio y el problema se leyeron sin errores de sintaxis (7 Predicates parsed, 4 Actions parsed, 6 Objects parsed).

Instanciación (Grounding): Se generaron 18 variables y 18 operadores a partir de los objetos concretos (base, r1, r2, ant, m1, m2), lo que demuestra que la estructura de tipos y predicados es totalmente coherente.

2. Rendimiento de la Búsqueda
Eficiencia óptima: El planificador resolvió la misión en tan solo 0.016 segundos y expandió únicamente 25 nodos (Search time: 0.016, 25 Nodes expanded). Esto indica que la heurística seleccionada guió al algoritmo de manera muy directa hacia la meta sin realizar exploraciones innecesarias.

Longitud del plan: El plan resultante consta de 9 acciones (Plan length: 9), lo cual representa una secuencia lógica y compacta para que el rover se desplace, recolecte, analice y transmita ambas muestras (m1 y m2).

3. Sobre la Advertencia del Validador
validate could not be found on the PATH so the plan can not be validated.

Qué significa: Esta advertencia indica que el ejecutable externo de validación de planes (VAL) no está instalado o no se encuentra registrado en las variables de entorno (PATH) de tu sistema.

Impacto: Ninguno sobre la solución. Pyperplan generó el plan correctamente por sí mismo; esta herramienta externa solo se suele usar opcionalmente para una doble verificación formal del plan generado.

